<a href="https://colab.research.google.com/github/mailrohitkg/Agentic-AI/blob/main/AgenticAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
pip install langchain langchain-openai langgraph tavily-python pydantic

In [16]:
import os
from google.colab import userdata

# Securely set API keys using Google Colab Secrets
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
os.environ["TAVILY_API_KEY"] = userdata.get('TAVILY_API_KEY')

In [18]:
import os
from pydantic import BaseModel, Field
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_openai import ChatOpenAI
from tavily import TavilyClient
import pandas as pd
from IPython.display import HTML, display
from google.colab import userdata

# Securely set API keys using Google Colab Secrets
os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
os.environ["TAVILY_API_KEY"] = userdata.get('TAVILY_API_KEY')

# Schema for Deep Microarchitecture Telemetry
class MicroarchitectureSpec(BaseModel):
    chip_name: str = Field(description="Official accelerator name")
    microarchitecture_codename: str = Field(description="Silicon codename/generation (e.g., Rubin, Blackwood, CDNA 5, Ironwood, Oryon)")
    peak_compute_performance_tflops: str = Field(description="Peak compute performance metrics, such as FP8/FP4/FP16 TFLOPS configurations")
    tdp_watts: str = Field(description="Thermal Design Power (TDP) in Watts (e.g., 1000W, 750W, 150W)")
    performance_per_watt: str = Field(description="Energy efficiency assessment (e.g., TFLOPS/Watt, performance improvements relative to predecessors under similar power targets)")
    die_topology_details: str = Field(description="Core microarchitectural details: layout, number of compute units/SMs/CUs, matrix processing units, vector engine configs, or internal caches")
    memory_type: str = Field(description="Memory technology used (e.g., HBM3e, HBM4, LPDDR5X)")
    memory_capacity: str = Field(description="Total memory capacity in GB (e.g., 288GB, 192GB)")
    memory_bandwidth: str = Field(description="Peak memory bandwidth (e.g., 8.0 TB/s, 4.8 TB/s)")
    on_chip_sram_cache: str = Field(description="Details about L1/L2/L3 caches, register files, or local scratchpad SRAM capacities on-die")
    execution_units_structure: str = Field(description="Structure of the executing cores (e.g., Tensor Cores, Matrix Engines, Custom ALUs, Vector Units)")
    scale_up_network: str = Field(description="Intra-node scale-up interconnect fabric (e.g., NVLink 5, Infinity Fabric, UALink)")
    scale_out_network: str = Field(description="Inter-node scale-out network fabric (e.g., InfiniBand, RoCE v2, Ultra Ethernet, ICI)")
    primary_use_cases: str = Field(description="Optimal models or use cases (e.g., Large LLM serving, Edge vision, Multi-modal MoE)")

tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))
model_eval = ChatOpenAI(model="gpt-4o", temperature=0.5)

targets = [
    "NVIDIA Rubin",
    "AMD Instinct MI400",
    "NVIDIA Blackwell Ultra B300",
    "Google TPU v8i",
    "Google TPU v8t",
    "AWS Trainium3",
    "Qualcomm Snapdragon AI250"
]

micro_specs = []

print("[*] Initiating deep-dive microarchitecture mining with compute performance and TDP metrics...")
for target in targets:
    # Dedicated microarchitecture and performance search
    query = f"{target} microarchitecture performance TFLOPS TDP Watts efficiency TDP memory capacity HBM bandwidth TB/s compute units execution units"
    search_res = tavily_client.search(query=query, max_results=3, search_depth="advanced")
    context_text = "\n\n".join([r["content"] for r in search_res.get("results", [])])

    # Structure
    extractor = model_eval.with_structured_output(MicroarchitectureSpec)
    prompt = f"""Extract microarchitecture and performance details for '{target}' using the context below. Focus on peak compute performance, TDP in Watts, energy efficiency, memory subsystem configurations, and physical compute units.\n\nContext:\n{context_text}"""

    try:
        spec = extractor.invoke([
            SystemMessage(content="You are an expert microprocessor engineer detailing silicon blocks, compute performance throughputs, TDP power envelopes, energy efficiency, and topologies."),
            HumanMessage(content=prompt)
        ])
        micro_specs.append(spec)
        print(f"[+] Completed performance, TDP & microarchitecture extraction for {target}")
    except Exception as e:
        print(f"[-] Error extracting for {target}: {e}")

# Convert to DataFrame and Display
micro_records = []
for spec in micro_specs:
    micro_records.append({
        "Accelerator": spec.chip_name,
        "Codename / Arch": spec.microarchitecture_codename,
        "Peak Performance": spec.peak_compute_performance_tflops,
        "TDP (Watts)": spec.tdp_watts,
        "Efficiency (Perf/Watt)": spec.performance_per_watt,
        "Memory Type": spec.memory_type,
        "Memory Capacity": spec.memory_capacity,
        "Memory Bandwidth": spec.memory_bandwidth,
        "Die Topology Details": spec.die_topology_details,
        "On-Chip SRAM / Cache": spec.on_chip_sram_cache,
        "Execution Unit Structure": spec.execution_units_structure,
        "Scale-Up Fabric": spec.scale_up_network,
        "Scale-Out Fabric": spec.scale_out_network,
        "Use Cases": spec.primary_use_cases
    })

micro_df = pd.DataFrame(micro_records)

styled_micro = micro_df.style.set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#1e3a8a'), ('color', '#ffffff'), ('font-weight', 'bold'), ('text-align', 'left'), ('padding', '10px'), ('font-family', 'sans-serif') ]},
    {'selector': 'td', 'props': [('padding', '10px'), ('border-bottom', '1px solid #e2e8f0'), ('font-family', 'sans-serif'), ('font-size', '12px') ]}
]).hide(axis='index').to_html()

display(HTML("<h3>🛠️ Deep-Dive Hardware Microarchitecture, TDP & Performance Specification Matrix</h3>"))
display(HTML(styled_micro))

[*] Initiating deep-dive microarchitecture mining with compute performance and TDP metrics...
[+] Completed performance, TDP & microarchitecture extraction for NVIDIA Rubin
[+] Completed performance, TDP & microarchitecture extraction for AMD Instinct MI400
[+] Completed performance, TDP & microarchitecture extraction for NVIDIA Blackwell Ultra B300
[+] Completed performance, TDP & microarchitecture extraction for Google TPU v8i
[+] Completed performance, TDP & microarchitecture extraction for Google TPU v8t
[+] Completed performance, TDP & microarchitecture extraction for AWS Trainium3
[+] Completed performance, TDP & microarchitecture extraction for Qualcomm Snapdragon AI250


Accelerator,Codename / Arch,Peak Performance,TDP (Watts),Efficiency (Perf/Watt),Memory Type,Memory Capacity,Memory Bandwidth,Die Topology Details,On-Chip SRAM / Cache,Execution Unit Structure,Scale-Up Fabric,Scale-Out Fabric,Use Cases
NVIDIA Rubin GPU,Rubin,"FP4: 50 PFLOPS, FP8: 17.5 PFLOPS, FP16: ~8,000 TFLOPS","~2,300 W",Up to 10x more agentic throughput per unit of energy than Blackwell,HBM4,288 GB,22 TB/s,"336 billion transistors, 224 streaming multiprocessors, 896 Tensor Cores, third-generation Transformer Engine",Not specified,"Tensor Cores, Transformer Engine","NVLink 6, 3.6 TB/s","NVLink-C2C, 1.8 TB/s","Agentic AI workloads, multitrillion-parameter models, high-concurrency inference"
AMD Instinct MI400,CDNA Next,"40 PFLOPS FP4, 20 PFLOPS FP8",1000W,Doubling compute capability compared to MI350,HBM4,432GB,19.6 TB/s,256 Work Group Processors (WGP) with Wave32 execution,192 MB Infinity Cache,Wave32 execution units,Ultra Accelerator Link (UALoF),Pensando Vulcano 800G,"Frontier-class AI models inference and training, large simulations, memory-intensive models"
NVIDIA Blackwell Ultra B300,Blackwell Ultra,"FP4: 15,000 TFLOPS, FP8: 7,000 TFLOPS, FP16: 3,500 TFLOPS","1,400W",~10.71 TFLOPS/Watt at FP4,HBM3e,288 GB,8 TB/s,"160 streaming multiprocessors, dual-die design linked by a 10 TB/s on-package interface, 20,480+ CUDA cores, 5th-generation Ultra Tensor Cores","Fifth-generation Tensor Cores with native NVFP4 support, second-gen Transformer engine","Ultra Tensor Cores, CUDA Cores",NVLink 5 (1.8 TB/s),ConnectX-8 (1.6T),"Large LLM serving, long-context reasoning, high-volume generative media, high-concurrency serving"
Google TPU v8i,Axion,10.1 PFLOPs (FP4),N/A,2x better perf/watt than Trillium,HBM3e,288 GB,8.6 TB/s,Single compute die and one I/O die with six stacks of HBM3e,384 MB of on-chip SRAM,Collectives Acceleration Engine (CAE),19.2 Tb/s inter-chip bandwidth,Boardfly (Dragonfly-inspired),Latency-Sensitive Inference
Google TPU v8t,Sunfish,"12,600 TFLOPs (FP4)",Not specified,2.7x–2.8x TCO improvement vs TPU 7x,HBM3e/4,216 GB,6.52 TB/s,Engineered for massive multi-megawatt pre-training pods with 3D Torus interconnect and high-density SerDes routing,"128 MB SRAM (""Vmem"")","Dense Matrix Throughput, Native FP4 Precision",3D Torus,Not specified,Frontier-Scale Pre-Training
AWS Trainium3,NeuronCore-v4,"2,517 TFLOPS (FP8/MXFP4), 671 TFLOPS (BF16/FP16/TF32), 183 TFLOPS (FP32)",~400W,Improved by over 4x compared to Trainium2,HBM3e,144 GB,4.9 TB/s,"Eight NeuronCore-v4 cores optimized for matrix multiplication, with on-chip scratchpad memory managed by the Neuron compiler.",256 MiB SBUF,"Dedicated NeuronCores optimized for matrix multiplication, supporting FP8, BF16, FP16, FP32 data types.",NeuronLink-v4 with 2.56 TB/sec bandwidth per device,NeuronLink-v4 for device-to-device interconnect,"Large-scale AI model training, Mixed-precision training, Efficient training of models with hundreds of billions of parameters."
Qualcomm Snapdragon AI250,HBC Gen 1,383 TFLOPS FP16/BF16,500W,6x higher bandwidth-per-watt than traditional HBM solutions,LPDDR (via HBC architecture),Not disclosed,133 TB/s,"110 Compute Units clocked at 1.7 GHz, utilizing HBC architecture with LPDDR DRAM stacks",200x the capacity of traditional on-chip SRAM,"AI accelerators beneath DRAM stacks, optimized for near-memory computation",Not disclosed,Not disclosed,"AI and ML workloads, optimized for transformer models and large tensor operations"


In [19]:
import os
from pydantic import BaseModel, Field
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_openai import ChatOpenAI
from tavily import TavilyClient
import pandas as pd
from IPython.display import HTML, display
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get('OPENAI_API_KEY')
os.environ["TAVILY_API_KEY"] = userdata.get('TAVILY_API_KEY')

tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))
model_eval = ChatOpenAI(model="gpt-4o", temperature=0.5)

class CostPerformanceAnalysis(BaseModel):
    chip_name: str = Field(description="Official accelerator name")
    estimated_unit_cost_usd: str = Field(description="Estimated single-chip price, subscription/cloud rate, or system rack TCO context")
    compute_performance_metric: str = Field(description="Selected performance baseline (e.g. FP8/FP4 TFLOPS, token latency, or training throughput)")
    cost_to_performance_ratio: str = Field(description="Synthesized cost-to-performance ratio or TCO analysis (e.g., FLOPS per dollar, training/inference efficiency)")
    strategic_financial_verdict: str = Field(description="Brief strategic verdict for a financial executive regarding this chip's acquisition feasibility")

targets = [
    "NVIDIA Rubin",
    "AMD Instinct MI400",
    "NVIDIA Blackwell Ultra B300",
    "Google TPU v8i",
    "Google TPU v8t",
    "AWS Trainium3",
    "Qualcomm Snapdragon AI250"
]

cost_perf_data = []

print("[*] Mining pricing and financial indicators for 2025/2026 fleet using gpt-4o...")
for target in targets:
    query = f"{target} estimated price cost per chip pricing subscription model system TCO cost performance ratio"
    search_res = tavily_client.search(query=query, max_results=3, search_depth="advanced")
    context_text = "\n\n".join([r["content"] for r in search_res.get("results", [])])

    extractor = model_eval.with_structured_output(CostPerformanceAnalysis)
    prompt = f"""Analyze the cost-to-performance ratio for '{target}' using the context below. Focus on estimated market price, financial TCO (total cost of ownership) vs. its architectural compute performance, and generate a clear cost-to-performance metric.\n\nContext:\n{context_text}"""

    try:
        analysis = extractor.invoke([
            SystemMessage(content="You are an expert financial and hardware performance analyst specializing in data center AI infrastructure budgeting."),
            HumanMessage(content=prompt)
        ])
        cost_perf_data.append(analysis)
        print(f"[+] Completed cost-to-performance modeling for {target}")
    except Exception as e:
        print(f"[-] Error modeling {target}: {e}")

# Prepare and display the final comparative matrix
records = []
for item in cost_perf_data:
    records.append({
        "Accelerator": item.chip_name,
        "Estimated Cost/TCO": item.estimated_unit_cost_usd,
        "Compute Performance Baseline": item.compute_performance_metric,
        "Cost-Performance Ratio": item.cost_to_performance_ratio,
        "Financial Verdict": item.strategic_financial_verdict
    })

cost_df = pd.DataFrame(records)
styled_cost = cost_df.style.set_table_styles([
    {'selector': 'th', 'props': [('background-color', '#065f46'), ('color', '#ffffff'), ('font-weight', 'bold'), ('text-align', 'left'), ('padding', '10px'), ('font-family', 'sans-serif') ]},
    {'selector': 'td', 'props': [('padding', '10px'), ('border-bottom', '1px solid #e2e8f0'), ('font-family', 'sans-serif'), ('font-size', '12px') ]}
]).hide(axis='index').to_html()

display(HTML("<h3>💰 2025/2026 Fleet Cost-to-Performance Ratio Analysis</h3>"))
display(HTML(styled_cost))

[*] Mining pricing and financial indicators for 2025/2026 fleet using gpt-4o...
[+] Completed cost-to-performance modeling for NVIDIA Rubin
[+] Completed cost-to-performance modeling for AMD Instinct MI400
[+] Completed cost-to-performance modeling for NVIDIA Blackwell Ultra B300
[+] Completed cost-to-performance modeling for Google TPU v8i
[+] Completed cost-to-performance modeling for Google TPU v8t
[+] Completed cost-to-performance modeling for AWS Trainium3
[+] Completed cost-to-performance modeling for Qualcomm Snapdragon AI250


Accelerator,Estimated Cost/TCO,Compute Performance Baseline,Cost-Performance Ratio,Financial Verdict
NVIDIA Rubin,$3.5 to $4.0 million per rack,60% of standard Rubin R200 compute performance,Higher FLOPs-per-dollar for prefill workloads due to reduced NVLink and HBM costs,"The NVIDIA Rubin offers a compelling cost-to-performance advantage for prefill workloads, making it an economically superior choice for tasks where compute is the bottleneck rather than data movement. The reduced manufacturing and operational costs, combined with the strategic avoidance of high-cost components like NVLink and HBM, make it a feasible acquisition for data centers focused on prefill tasks. However, it may not be suitable for decode workloads requiring high bandwidth and real-time processing."
AMD Instinct MI455X,"$30,000",40 PFLOPS FP4,1.33 TFLOPS per dollar,"The AMD Instinct MI455X offers a compelling cost-to-performance ratio with its 40 PFLOPS of FP4 performance at an estimated $30,000, translating to 1.33 TFLOPS per dollar. This positions it as a highly competitive option against NVIDIA's offerings, particularly for organizations with standard workloads that can leverage AMD's hardware discount to offset potential additional engineering costs. For data centers prioritizing TCO and performance efficiency, the MI455X is a strategically sound investment."
NVIDIA Blackwell Ultra B300,"$40,000 - $50,000 per unit","~150,000+ tokens/sec FP4",0.34x relative cost-per-token (66% cheaper than baseline H100 SXM),"The B300 offers a compelling cost-to-performance advantage for large-scale inference tasks, particularly when high throughput and memory capacity are critical. Despite its higher upfront cost, the 66% cheaper cost-per-token compared to the H100 SXM makes it a strategic investment for data centers focused on maximizing inference efficiency, especially for models requiring extensive VRAM and FP4 compute. However, for smaller models or cost-sensitive deployments, the H200 may remain more economical. Evaluate current pricing dynamics and specific workload requirements before acquisition."
Google TPU v8i,$5-7 per chip-hour,Tokens per second (estimated),Up to 80% performance-per-dollar improvement over previous generation,"The TPU v8i is a strategically sound investment for AI workloads requiring high throughput and cost efficiency. Despite the lack of independent benchmarks, the anticipated performance-per-dollar improvement and competitive pricing make it a viable option for reducing operational costs compared to alternatives like the B200 and B300, especially for high concurrency and token throughput scenarios."
Google TPU v8t,$5-7 per chip-hour,Performance-per-dollar improvement (2.7x over TPU 7x for training),2.7x improvement in performance-per-dollar for large-scale training,"The TPU v8t offers a significant performance-per-dollar advantage over previous generations, making it a strong candidate for organizations focused on large-scale AI training. The estimated cost aligns with market expectations for high-performance chips, suggesting a favorable TCO for enterprises looking to scale efficiently."
AWS Trainium3,$1 per chip per hour (on-demand),Cost per token trained,"Trainium3 delivers 40-47% cost savings on large models compared to H100, reducing cost per token significantly","AWS Trainium3 is a cost-effective alternative for large-scale AI training workloads, offering significant savings over NVIDIA H100, particularly under long-term contract pricing. It is a strategically sound investment for organizations prioritizing cost-efficiency in AI training."
Qualcomm Snapdragon AI250,"$6,150 per unit",FP4 Petaflops per Rack,"$2,604 per petaflops per rack","The Qualcomm Snapdragon AI250 presents a compelling cost-to-performance ratio, especially when compared to Nvidia's offerings. With a cost of $2,604 per petaflop per rack, it offers a superior TCO and performance efficiency, making it an economically viable option for large-scale AI deployments